In [ ]:
# --- Librerie esterne ---
# Grafici.
import matplotlib.pyplot as plt
# Calcolo numerico su array.
import numpy as np
# Percorsi dei file e creazione di cartelle.
import os
# Per nascondere gli avvisi non importanti delle librerie.
import warnings
# Tabelle (DataFrame).
import pandas as pd
# PyTorch: qui serve solo per sapere se c'è una GPU.
import torch
# UMAP: proiezione 2D delle molecole per la figura dello split.
import umap
# RDKit: lettura delle molecole e similarità tra fingerprint.
from rdkit import Chem, DataStructs
# Mostra nel notebook le immagini salvate su disco.
from IPython.display import Image
# RDKit: disegno delle molecole e descrittori (celle finali).
from rdkit.Chem import Draw, Descriptors, Crippen

# --- Moduli della pipeline, in ordine di esecuzione (file s0 … s7) ---
# Passo 0: nomi delle isoforme e delle colonne target.
from s0_config import ISOFORMS, TARGET_COLS
# Passo 1: split per gruppi chimici e fingerprint.
from s1_split import splitter, generate_fingerprints, fps_to_numpy
# Passo 2: metriche e grafici di valutazione.
from s2_metrics import calculate_metrics, per_isoform_report, parity_plots, ranking_metrics_per_isoform, r2_comparison_plot
# Passo 3: baseline LightGBM.
from s3_baseline import run_lgbm_baseline
# Passo 5: training del D-MPNN, scelta del seed e predizione (il passo 4 è usato al suo interno).
from s5_training import train_dmpnn, select_best_seed, seed_view, predict_dmpnn
# Passo 6: stacking LightGBM + D-MPNN.
from s6_stacking import fit_stacking_meta_learner, predict_stacking
# Passo 7: bundle di inferenza e predizione su molecole nuove.
from s7_inference import export_bundle, load_bundle, predict, predict_to_dataframe

In [ ]:
# Nasconde gli avvisi (warning) delle librerie per un output più pulito.
warnings.filterwarnings('ignore')

# Cartella in cui vengono scritti tutti i risultati.
OUTPUT_DIR = 'outputs'
# La crea se non esiste.
os.makedirs(OUTPUT_DIR, exist_ok=True)
# Una sottocartella per ogni fase della pipeline.
SPLIT_OUT, BASELINE_OUT, DMPNN_OUT, STACKING_OUT, COMPARISON_OUT, SCREENING_OUT, MODEL_OUT = (
    os.path.join(OUTPUT_DIR, n) for n in
    ('split', 'baseline', 'dmpnn', 'stacking', 'comparison', 'screening', 'model'))
# Le crea se non esistono.
for _d in (SPLIT_OUT, BASELINE_OUT, DMPNN_OUT, STACKING_OUT, COMPARISON_OUT, SCREENING_OUT, MODEL_OUT):
    os.makedirs(_d, exist_ok=True)
# Usa la GPU se disponibile, altrimenti la CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Mostra il dispositivo scelto.
print('device:', device)

# Stile comune di tutti i grafici.
plt.rcParams['figure.figsize'] = (13.33, 4)  # dimensione di default, in pollici
plt.rcParams['figure.dpi'] = 100  # risoluzione a schermo (bassa: notebook leggero)
plt.rcParams['savefig.dpi'] = 600  # risoluzione dei file salvati (alta: per la pubblicazione)
plt.rcParams['font.size'] = 12  # dimensione del testo
plt.rcParams['axes.titlesize'] = 16  # dimensione dei titoli
plt.rcParams['axes.labelsize'] = 14  # dimensione delle etichette degli assi
plt.rcParams['legend.fontsize'] = 16  # dimensione del testo della legenda
plt.rcParams['lines.linewidth'] = 2  # spessore delle linee
plt.rcParams['lines.markersize'] = 4  # dimensione dei marcatori
plt.rcParams['axes.grid'] = False  # nessuna griglia di default
plt.rcParams['grid.alpha'] = 0.5  # trasparenza della griglia, quando attivata
plt.rcParams['grid.linestyle'] = '-.' # stile tratto-punto della griglia

In [ ]:
# Legge il dataset: una riga per molecola, lo SMILES e le 9 colonne di pIC50.
df = pd.read_excel('data.xlsx')
# Dimensioni (righe, colonne).
print('shape:', df.shape)

# Parametri del clustering, usati solo se lo split va ricalcolato.
UMAP_N_NEIGHBORS = 30  # vicini considerati da UMAP
UMAP_MIN_DIST = 0.25  # distanza minima tra punti nella proiezione
UMAP_N_COMPONENTS = 2  # dimensioni della proiezione
HDBSCAN_MIN_CLUSTER_SIZE = 5  # molecole minime per formare un gruppo denso
HDBSCAN_MIN_SAMPLES = 1  # più è basso, meno molecole vengono considerate rumore

# Cartella dello split di riferimento: è salvato apposta, perché UMAP e HDBSCAN danno
# risultati diversi con versioni diverse delle librerie.
SPLIT_DIR = 'split'
# Percorsi dei tre CSV attesi.
split_csv = [os.path.join(SPLIT_DIR, f'{s}_set_multitarget.csv') for s in ('train', 'val', 'test')]

# Se lo split di riferimento esiste…
if all(os.path.exists(f) for f in split_csv):
    # …lo carica…
    train_df, val_df, test_df = (pd.read_csv(f) for f in split_csv)
    # …e lo segnala.
    print(f'split loaded from {SPLIT_DIR}/ (reference partition of the published run)')
# Altrimenti lo ricalcola da zero (passo 1).
else:
    # Avviso.
    print(f'{SPLIT_DIR}/ not found: recomputing the split (UMAP + HDBSCAN)')
    # Split per gruppi chimici con i parametri definiti sopra.
    train_df, val_df, test_df = splitter(df, output_dir=SPLIT_OUT,
                                         umap_n_neighbors=UMAP_N_NEIGHBORS,  # parametri UMAP
                                         umap_min_dist=UMAP_MIN_DIST,
                                         umap_n_components=UMAP_N_COMPONENTS,
                                         hdbscan_min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,  # parametri HDBSCAN
                                         hdbscan_min_samples=HDBSCAN_MIN_SAMPLES)
# Dimensioni dei tre insiemi.
print(f'Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}')

# Train e validation uniti: è l'insieme su cui si fa la cross-validation.
cv_df = pd.concat([train_df, val_df]).reset_index(drop=True)
# Misure disponibili per isoforma (le isoforme rare ne hanno poche centinaia).
print('target counts (train+val):', dict(zip(ISOFORMS, cv_df[TARGET_COLS].notna().sum().tolist())))

# Dimensione di ogni gruppo chimico, dal più grande al più piccolo.
group_sizes = cv_df['Cluster_Butina'].value_counts()
# Numero di gruppi e peso del gruppo più grande.
print(f'cv: {group_sizes.size} groups | largest cluster = {group_sizes.iloc[0]} '
      f'({100 * group_sizes.iloc[0] / len(cv_df):.1f}% of cv)')

In [ ]:
# Parametri della mappa UMAP di questa figura (diversi da quelli dello split).
VIZ_N_NEIGHBORS = 15  # vicini considerati
VIZ_MIN_DIST = 0.1  # distanza minima tra punti
SEED = 42  # seme per la riproducibilità

# Tutte le molecole in una tabella, con una colonna che dice in quale insieme sono.
split_df = pd.concat([train_df.assign(Split='Train'),  # molecole di training
                      val_df.assign(Split='Val'),  # di validazione
                      test_df.assign(Split='Test')],  # di test
                     ignore_index=True)  # indice rinumerato da 0

# Fingerprint Morgan di tutte le molecole e indici di quelle valide.
fps, ok_idx = generate_fingerprints(split_df['Smiles'].astype(str).tolist())
# Tiene solo le righe valide.
split_df = split_df.iloc[ok_idx].reset_index(drop=True)
# Insieme di appartenenza di ogni molecola ('Train', 'Val' o 'Test').
splits = split_df['Split'].values
# Numero di molecole.
n_mol = len(fps)
# Matrice dei bit (molecole × 2048), input di UMAP.
X_fp = fps_to_numpy(fps).astype(np.uint8)

# Avviso: UMAP su ~7500 molecole richiede circa un minuto.
print(f'UMAP (Jaccard) on {n_mol} molecules...')
# Proiezione 2D con distanza di Jaccard (= 1 - Tanimoto); n_jobs=1 per la riproducibilità.
emb = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS, min_dist=VIZ_MIN_DIST,
                metric='jaccard', random_state=SEED, n_jobs=1).fit_transform(X_fp)

# Split casuale di confronto, con le stesse dimensioni di quello per gruppi.
n_te, n_va = int((splits == 'Test').sum()), int((splits == 'Val').sum())  # dimensioni di test e validation
perm = np.random.default_rng(SEED).permutation(n_mol)  # ordine casuale delle molecole
splits_rnd = np.empty(n_mol, dtype=object)  # etichette dello split casuale
splits_rnd[perm[:n_te]] = 'Test'  # le prime n_te vanno in test
splits_rnd[perm[n_te:n_te + n_va]] = 'Val'  # le successive n_va in validation
splits_rnd[perm[n_te + n_va:]] = 'Train'  # il resto in training


# Mediana, sulle molecole di test, della similarità di Tanimoto con la molecola più simile
# di train+val: alta = il test somiglia al training (split facile), bassa = split difficile.
def nn_median(sv):
    # Fingerprint di train+val.
    fit = [fps[i] for i in np.where((sv == 'Train') | (sv == 'Val'))[0]]
    # Indici delle molecole di test.
    te = np.where(sv == 'Test')[0]
    # Per ogni molecola di test, la similarità massima con train+val.
    nn = np.array([max(DataStructs.BulkTanimotoSimilarity(fps[i], fit)) for i in te])
    # Mediana di questi valori.
    return float(np.median(nn))


# Mediana per lo split per gruppi e per quello casuale.
nn_clu, nn_rnd = nn_median(splits), nn_median(splits_rnd)

# Stile dei punti di ogni insieme: colore, dimensione, trasparenza, ordine di disegno, bordo.
STYLE = {
    'Train': dict(c='#9bc1bc', s=40, alpha=0.50, zorder=1, linewidths=0),  # verde chiaro, sotto
    'Val': dict(c='#f4f1bb', s=40, alpha=0.85, zorder=2, linewidths=0.5, edgecolors='white'),  # giallo
    'Test': dict(c='#ed6a5a', s=40, alpha=0.90, zorder=3, linewidths=0.6, edgecolors='white'),  # rosso, sopra
}


# Disegna in un riquadro la mappa UMAP colorata secondo l'assegnazione sv.
def draw_split(ax, sv, title):
    # Per ogni insieme…
    for sp in ['Train', 'Val', 'Test']:
        # …molecole che vi appartengono…
        m = sv == sp
        # …disegnate con il loro stile (rasterized = file più leggero).
        ax.scatter(emb[m, 0], emb[m, 1], label=sp, rasterized=True, **STYLE[sp])
    # Titolo del riquadro.
    ax.set_title(title, fontsize=17, fontweight='bold', pad=14, color='#1f1f1f')
    # Legenda con riquadro bianco e marcatori ingranditi.
    leg = ax.legend(loc='upper left', fontsize=17, frameon=True, borderpad=1.0,
                    handletextpad=0.8, labelspacing=0.85, markerscale=2.6, fancybox=True)
    leg.get_frame().set_facecolor('white')  # sfondo della legenda
    leg.get_frame().set_edgecolor('#dcdcdc')  # bordo grigio chiaro
    leg.get_frame().set_linewidth(0.8)  # spessore del bordo
    leg.get_frame().set_alpha(0.92)  # leggera trasparenza
    # Tacche degli assi grigie e piccole.
    ax.tick_params(colors='#888888', labelsize=11, length=4)
    # Piccolo margine attorno ai punti.
    ax.margins(0.03)
    # Nasconde i bordi superiore e destro…
    for s in ['top', 'right']:
        # …del riquadro.
        ax.spines[s].set_visible(False)
    # Colora di grigio i bordi sinistro e inferiore…
    for s in ['left', 'bottom']:
        # …del riquadro.
        ax.spines[s].set_color('#bbbbbb')
    # Mostra le etichette degli assi solo sui riquadri esterni.
    ax.label_outer()


# Figura con due riquadri affiancati che condividono gli assi.
fig, (axL, axR) = plt.subplots(1, 2, figsize=(18, 8.5), sharex=True, sharey=True, constrained_layout=True)
# A sinistra lo split casuale…
draw_split(axL, splits_rnd, 'Random split')
# …a destra quello per gruppi.
draw_split(axR, splits, 'Cluster-based split')
# Salva l'immagine con sfondo trasparente.
fig.savefig(os.path.join(SPLIT_OUT, 'umap_split_visualization.png'), dpi=300, bbox_inches='tight', transparent=True)
# Mostra la figura nel notebook.
plt.show()

# Confronto numerico delle due similarità mediane.
print(f'median NN-sim test->train | cluster-based={nn_clu:.3f}  random={nn_rnd:.3f}')

In [ ]:
# Passo 3: baseline LightGBM (fit su train+val, valutazione sul test, predizioni OOF per lo stacking).
lgbm_res = run_lgbm_baseline(train_df, val_df, test_df, BASELINE_OUT)

In [ ]:
# Tutte le molecole insieme.
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
# Ogni molecola diventa un gruppo a sé: così anche la CV interna della baseline è casuale.
all_df['Cluster_Butina'] = np.arange(len(all_df))

# Ordine casuale riproducibile delle molecole.
perm_rnd = np.random.default_rng(42).permutation(len(all_df))
# Stesse dimensioni di test e validation dello split per gruppi.
n_te, n_va = len(test_df), len(val_df)
# Test casuale: le prime n_te molecole dell'ordine casuale.
rnd_test = all_df.iloc[perm_rnd[:n_te]].reset_index(drop=True)
# Validation casuale: le successive n_va.
rnd_val = all_df.iloc[perm_rnd[n_te:n_te + n_va]].reset_index(drop=True)
# Training casuale: tutte le altre.
rnd_train = all_df.iloc[perm_rnd[n_te + n_va:]].reset_index(drop=True)
# Conferma delle dimensioni.
print(f'random split: Train={len(rnd_train)} Val={len(rnd_val)} Test={len(rnd_test)} '
      f'(same sizes as the cluster-based split)')

# Stessa baseline, ma sullo split casuale (risultati in una sottocartella).
lgbm_rnd = run_lgbm_baseline(rnd_train, rnd_val, rnd_test, os.path.join(SPLIT_OUT, 'random_split_baseline'))

# Tabella di confronto tra i due split.
cmp_df = pd.DataFrame([
    {'split': 'random', 'NN_sim_median': nn_rnd,  # split casuale
     'R2_pooled': lgbm_rnd['test_metrics']['r2'], 'MSE': lgbm_rnd['test_metrics']['mse'],
     'Spearman': lgbm_rnd['test_metrics']['spearman_r']},
    {'split': 'cluster-based', 'NN_sim_median': nn_clu,  # split per gruppi
     'R2_pooled': lgbm_res['test_metrics']['r2'], 'MSE': lgbm_res['test_metrics']['mse'],
     'Spearman': lgbm_res['test_metrics']['spearman_r']},
])
# Salva la tabella.
cmp_df.to_csv(os.path.join(SPLIT_OUT, 'split_difficulty.csv'), index=False)
# La mostra arrotondata a 3 decimali.
display(cmp_df.round(3))
# Di quanto lo split casuale "gonfia" l'R².
delta = lgbm_rnd['test_metrics']['r2'] - lgbm_res['test_metrics']['r2']
# Stampa il risultato.
print(f'\nR2 inflated by the random split: {delta:+.3f}  (median NN-sim {nn_rnd:.2f} against {nn_clu:.2f})')

# Grafico a barre dei due R².
fig, ax = plt.subplots(figsize=(7, 4.5))
# Valori da disegnare.
vals = [lgbm_rnd['test_metrics']['r2'], lgbm_res['test_metrics']['r2']]
# Due barre: grigia (casuale) e rossa (per gruppi).
ax.bar(['random', 'cluster-based'], vals, color=['#bdbdbd', '#ed6a5a'], edgecolor='black', width=0.55)
# Scrive il valore sopra ogni barra.
for i, v in enumerate(vals):
    # Testo centrato sulla barra i.
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=13)
# Etichetta dell'asse y.
ax.set_ylabel('Test R2 pooled (baseline LightGBM)')
# Titolo.
ax.set_title('Same model, two splitting criteria')
# Griglia orizzontale leggera.
ax.grid(True, alpha=0.3, axis='y')
# Sistema gli spazi.
plt.tight_layout()
# Salva l'immagine.
plt.savefig(os.path.join(SPLIT_OUT, 'split_difficulty.png'), dpi=300, bbox_inches='tight')
# La mostra.
plt.show()

In [ ]:
# Numero di seed: il training completo viene ripetuto N_SEEDS volte con inizializzazioni diverse.
N_SEEDS = 2

# Iperparametri dell'architettura D-MPNN (passo 4).
DMPNN_KWARGS = dict(hidden_dim=160,  # dimensione degli stati nascosti
                    num_layers=4,  # passi di message passing
                    conv_dropout=0.05,  # dropout nel message passing
                    set2set_steps=3,  # iterazioni di Set2Set nella lettura del grafo
                    fc1_hidden=192,  # neuroni del primo strato della testa
                    fc2_hidden=96,  # neuroni del secondo strato della testa
                    fc_dropout=0.3)  # dropout nella testa

# Iperparametri del training (passo 5).
TRAIN_KWARGS = dict(k_folds=5,  # fold della cross-validation per gruppi
                    epochs=120,  # epoche massime per fold
                    patience=18,  # epoche senza miglioramento prima dell'early stopping
                    batch_size=32,  # molecole per batch
                    lr=5e-4,  # learning rate di AdamW
                    weight_decay=0.01,  # regolarizzazione dei pesi
                    grad_clip=1.0,  # norma massima dei gradienti
                    edge_drop_prob=0.10,  # augmentation: probabilità di togliere un legame
                    node_mask_prob=0.05,  # augmentation: probabilità di azzerare un atomo
                    swa_start_frac=0.75,  # lo SWA parte al 75% delle epoche (ep. 90)
                    swa_lr=1e-4,  # learning rate durante lo SWA
                    fold_n_trials=30,  # tentativi per scegliere i fold più bilanciati
                    fold_seed=42,  # seme dei fold (uguali per tutti i seed)
                    num_workers=4)  # processi che preparano i batch

# pIC50 veri del test (molecole × 9).
test_y = test_df[TARGET_COLS].values.astype(np.float32)
# pIC50 veri di train+val, nello stesso ordine delle predizioni OOF.
Y_cv = cv_df[TARGET_COLS].values.astype(np.float32)
# R² della baseline sul test, come riferimento.
LGBM_TEST_R2 = lgbm_res['test_metrics']['r2']
# Riepilogo della configurazione.
print(f"[config] N_SEEDS={N_SEEDS}  EPOCHS={TRAIN_KWARGS['epochs']}  K_FOLDS={TRAIN_KWARGS['k_folds']}  "
      f"|  LGBM baseline R2={LGBM_TEST_R2:.4f}")

In [ ]:
# Passo 5: training in cross-validation per tutti i seed (richiede ore su GPU).
res_all = train_dmpnn(cv_df, N_SEEDS, DMPNN_KWARGS, 'pretrained/ssl_encoder_dmpnn.pt', DMPNN_OUT, device,
                      **TRAIN_KWARGS)

# Il seed viene scelto solo sulla cross-validation (R² OOF); il test viene soltanto valutato.
best_seed, seed_cv_r2 = select_best_seed(res_all, Y_cv)
# Predizioni sul test di ogni seed, per mostrare quanto variano da un seed all'altro.
seed_test_preds = [predict_dmpnn(seed_view(res_all, s), test_df['Smiles'].tolist(), device,
                                 batch_size=TRAIN_KWARGS['batch_size']) for s in range(N_SEEDS)]
# R² sul test di ogni seed.
seed_test_r2 = [calculate_metrics(p, test_y)['r2'] for p in seed_test_preds]

# Il seed scelto: da qui in poi tutte le celle usano solo questo.
res_dmpnn = seed_view(res_all, best_seed)
# Sue predizioni sul test.
dmpnn_test_preds = seed_test_preds[best_seed]
# Sue metriche sul test.
m_te_dmpnn = calculate_metrics(dmpnn_test_preds, test_y)

# Riepilogo: seed scelto e suo R² di cross-validation…
print(f'\nD-MPNN  seed {best_seed}, chosen on CV (OOF R² {seed_cv_r2[best_seed]:.3f})')
# …metriche sul test, con la baseline come riferimento…
print(f'  test  R² {m_te_dmpnn["r2"]:.3f}  MSE {m_te_dmpnn["mse"]:.3f}  '
      f'Spearman {m_te_dmpnn["spearman_r"]:.3f}   (LightGBM R² {LGBM_TEST_R2:.3f})')
# …e, con più seed, quanto variano gli altri sul test.
if N_SEEDS > 1:
    # R² sul test degli altri seed.
    others = '  '.join(f'seed {s} {r:.3f}' for s, r in enumerate(seed_test_r2) if s != best_seed)
    # Distanza tra il seed migliore e il peggiore sul test.
    print(f'  other seeds on test: {others}  ->  spread Δ {max(seed_test_r2) - min(seed_test_r2):.3f}')

# Salva R² di CV e di test di ogni seed.
pd.DataFrame({'seed': range(N_SEEDS), 'cv_oof_r2': seed_cv_r2, 'test_r2': seed_test_r2}).to_csv(
    os.path.join(DMPNN_OUT, 'seed_summary.csv'), index=False)
# Tabella e grafico per isoforma del seed scelto.
per_isoform_report(test_y, dmpnn_test_preds, output_path=os.path.join(DMPNN_OUT, 'per_isoform_test.csv'))
# Grafici di parità del seed scelto.
parity_plots(test_y, dmpnn_test_preds, output_path=os.path.join(DMPNN_OUT, 'parity_test.png'))
# Mostra i grafici appena salvati.
display(Image(filename=os.path.join(DMPNN_OUT, 'per_isoform_test.png')))
display(Image(filename=os.path.join(DMPNN_OUT, 'parity_test.png')))

In [ ]:
# Valori della penalità del Ridge tra cui lo stacking sceglie (con una CV interna).
ALPHA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]

# Predizioni out-of-fold dei due modelli su train+val: sono i dati di allenamento dello stacking.
oof_dict = {'LGBM': lgbm_res['oof'], 'DMPNN': res_dmpnn['oof']}
# Predizioni dei due modelli sul test.
test_dict = {'LGBM': lgbm_res['test_preds'], 'DMPNN': dmpnn_test_preds}
# Passo 6: un Ridge per isoforma, allenato sulle OOF.
meta = fit_stacking_meta_learner(oof_dict, Y_cv, alpha_grid=ALPHA_GRID, cv_splits=5)
# Predizioni combinate sul test.
stacked_preds = predict_stacking(meta, test_dict)
# Metriche dello stacking sul test.
stack_metrics = calculate_metrics(stacked_preds, test_y)
# Riepilogo.
print(f"[stacking LGBM+DMPNN, weights>=0]  Test R2={stack_metrics['r2']:.4f}  "
      f"MSE={stack_metrics['mse']:.4f}  Spearman={stack_metrics['spearman_r']:.4f}")

# Tabella e grafico per isoforma dello stacking.
per_isoform_report(test_y, stacked_preds, output_path=os.path.join(STACKING_OUT, 'per_isoform_test.csv'))
# Grafici di parità dello stacking.
parity_plots(test_y, stacked_preds, output_path=os.path.join(STACKING_OUT, 'parity_test.png'))
# Mostra i grafici appena salvati.
display(Image(filename=os.path.join(STACKING_OUT, 'per_isoform_test.png')))
display(Image(filename=os.path.join(STACKING_OUT, 'parity_test.png')))

In [ ]:
# Predizioni sul test dei tre modelli da confrontare.
all_test_preds = {'LGBM': lgbm_res['test_preds'],  # baseline
                  'DMPNN': dmpnn_test_preds,  # D-MPNN, seed scelto
                  'Stacking': stacked_preds}  # combinazione dei due
# Sotto questa soglia di molecole di test l'R² di un'isoforma è poco affidabile.
UNDERPOWERED_N = 25


# Tabella completa per isoforma: metriche di regressione, EF@10% e una riga finale di medie.
def full_per_isoform_table(y_true, y_pred):
    # Metriche di regressione per isoforma.
    reg = per_isoform_report(y_true, y_pred)
    # Metriche di ranking per isoforma.
    rnk = ranking_metrics_per_isoform(y_true, y_pred)
    # Colonne di regressione nell'ordine voluto.
    tab = reg[['isoform', 'n', 'r2', 'rmse', 'mae', 'pearson_r', 'spearman_r']].copy()
    # Aggiunge l'Enrichment Factor al 10%.
    tab['EF@10%'] = rnk['EF@10pc'].values
    # Nomi delle colonne più leggibili.
    tab.columns = ['Isoform', 'n', 'R2', 'RMSE', 'MAE', 'Pearson', 'Spearman', 'EF@10%']
    # Segnala le isoforme con poche molecole di test.
    tab['n<25'] = tab['n'] < UNDERPOWERED_N
    # Riga finale: totale delle molecole…
    mean_row = {'Isoform': 'MEAN', 'n': int(tab['n'].sum()), 'n<25': False}
    # …e media di ogni metrica sulle isoforme (ignorando i NaN).
    for c in ['R2', 'RMSE', 'MAE', 'Pearson', 'Spearman', 'EF@10%']:
        # Media della colonna c.
        mean_row[c] = float(np.nanmean(tab[c].values))
    # Tabella con la riga delle medie in fondo.
    return pd.concat([tab, pd.DataFrame([mean_row])], ignore_index=True)


# Tabelle complete dei tre modelli.
per_iso_full = {}
# Per ogni modello…
for name, preds in all_test_preds.items():
    # …calcola la tabella…
    t = full_per_isoform_table(test_y, preds)
    # …la conserva…
    per_iso_full[name] = t
    # …la salva…
    t.round(4).to_csv(os.path.join(COMPARISON_OUT, f'per_isoform_{name.lower()}.csv'), index=False)
    # …e la mostra con un titolo.
    print(f'\n================  PER-ISOFORM (test) - {name}  ================')
    # Tabella arrotondata a 3 decimali.
    display(t.round(3))

# Tabella riassuntiva (letta da figures.ipynb): una riga per isoforma e numero di molecole di test.
consolidated = pd.DataFrame({'isoform': ISOFORMS, 'n_test': per_iso_full['Stacking']['n'].values[:len(ISOFORMS)]})
# Per ogni modello…
for name in all_test_preds:
    # …le sue 9 righe (senza la riga delle medie)…
    src = per_iso_full[name].iloc[:len(ISOFORMS)]
    # …R², MAE e Spearman come colonne con il nome del modello.
    consolidated[f'R2_{name}'] = src['R2'].values  # R² del modello
    consolidated[f'MAE_{name}'] = src['MAE'].values  # errore assoluto medio
    consolidated[f'Spearman_{name}'] = src['Spearman'].values  # correlazione di rango
# Salva la tabella riassuntiva.
consolidated.to_csv(os.path.join(COMPARISON_OUT, 'per_isoform_consolidated.csv'), index=False)

# Istogramma dell'R² per isoforma: baseline, D-MPNN e stacking affiancati.
r2_comparison_plot(consolidated, os.path.join(COMPARISON_OUT, 'r2_per_isoform_comparison.png'),
                   underpowered_n=UNDERPOWERED_N)
# Lo mostra nel notebook.
display(Image(filename=os.path.join(COMPARISON_OUT, 'r2_per_isoform_comparison.png')))

# Tabelle e grafici di ranking per modello (letti da figures.ipynb).
for name, preds in all_test_preds.items():
    # Salva ranking_<modello>_test.csv e il grafico accanto.
    ranking_metrics_per_isoform(test_y, preds, output_path=os.path.join(COMPARISON_OUT, f'ranking_{name.lower()}.csv'))

# Molecole di test per isoforma.
n_test = consolidated['n_test'].values
# Isoforme con poche molecole di test.
underpowered = [iso for iso, n in zip(ISOFORMS, n_test) if n < UNDERPOWERED_N]
# Le segnala.
print(f'\nUNDERPOWERED isoforms (n<{UNDERPOWERED_N}, R2 not to be over-interpreted): {underpowered}')
# Riepilogo: medie sulle isoforme, con e senza quelle poco numerose.
print('Macro summary (mean over isoforms)  [all | n>=25 only]:')
# Per ogni modello…
for name in all_test_preds:
    # …le sue 9 righe…
    t = per_iso_full[name].iloc[:len(ISOFORMS)]
    # …solo le isoforme con abbastanza molecole…
    powered = t[t['n'] >= UNDERPOWERED_N]
    # …e la riga di riepilogo.
    print(f'  {name:9s}  R2_all={np.nanmean(t["R2"]):.4f}  R2_pow={np.nanmean(powered["R2"]):.4f}  '
          f'Spearman={np.nanmean(t["Spearman"]):.4f}  EF@10%={np.nanmean(t["EF@10%"]):.4f}')

In [ ]:
# R² di cross-validation del D-MPNN (seed scelto), un valore per fold.
cv_r2 = np.array([m['r2'] for m in res_dmpnn['fold_metrics']], dtype=float)

# Una riga per modello: R² di CV (solo D-MPNN) e metriche sul test.
rows = [
    {'model': 'LightGBM (Morgan FP) [baseline]', 'r2_cv_mean': np.nan, 'r2_cv_std': np.nan,
     'r2_test': lgbm_res['test_metrics']['r2'], 'mse_test': lgbm_res['test_metrics']['mse'],
     'spearman_test': lgbm_res['test_metrics']['spearman_r']},
    {'model': f'D-MPNN (seed {best_seed}, tgt_std+pretrain, uncertainty)', 'r2_cv_mean': cv_r2.mean(),
     'r2_cv_std': cv_r2.std(), 'r2_test': m_te_dmpnn['r2'], 'mse_test': m_te_dmpnn['mse'],
     'spearman_test': m_te_dmpnn['spearman_r']},
    {'model': 'Stacking LightGBM + D-MPNN', 'r2_cv_mean': np.nan, 'r2_cv_std': np.nan,
     'r2_test': stack_metrics['r2'], 'mse_test': stack_metrics['mse'], 'spearman_test': stack_metrics['spearman_r']},
]
# Tabella finale.
final_df = pd.DataFrame(rows)
# Salvata in CSV.
final_df.to_csv(os.path.join(COMPARISON_OUT, 'final_comparison.csv'), index=False)
# Mostrata come output della cella.
final_df

In [ ]:
# Isoforme di Classe I (il bersaglio di interesse)…
CLASS_I = ['HDAC1_HUMAN', 'HDAC2_HUMAN', 'HDAC3_HUMAN', 'HDAC8_HUMAN']
# …e tutte le altre.
OTHERS = ['HDAC4_HUMAN', 'HDAC5_HUMAN', 'HDAC6_HUMAN', 'HDAC7_HUMAN', 'HDAC9_HUMAN']


# Punteggio di selettività per la Classe I: media dei pIC50 di Classe I meno il massimo
# delle altre isoforme. Alto = la molecola colpisce la Classe I molto più delle altre.
def selectivity_score(df):
    # Media dei pIC50 di Classe I.
    mean_i = df[CLASS_I].mean(axis=1)
    # Massimo dei pIC50 delle altre isoforme.
    max_o = df[OTHERS].max(axis=1)
    # Restituisce le due componenti e il punteggio.
    return mean_i, max_o, mean_i - max_o


# Percorso del bundle di inferenza.
bundle_path = os.path.join(MODEL_OUT, 'inference_bundle.pt')
# Passo 7: salva modelli D-MPNN del seed scelto, LightGBM e stacking in un unico file.
export_bundle(bundle_path, res_dmpnn, meta, lgbm_res['models'])
# Lo ricarica, come farebbe un utente esterno.
bundle = load_bundle(bundle_path, map_location=device)
# Nomi delle colonne predette.
PRED_COLS = bundle['target_cols']

# Rilegge il dataset completo.
df_full = pd.read_excel('data.xlsx')
# Predice tutte le 9 isoforme di tutte le molecole.
preds_all = predict(df_full['Smiles'].astype(str).tolist(), bundle, device=device)
# Per ogni isoforma…
for j, col in enumerate(TARGET_COLS):
    # …valori misurati (copia modificabile)…
    vals = df_full[col].to_numpy(dtype=float, copy=True)
    # …celle non misurate…
    nan_mask = np.isnan(vals)
    # …riempite con la predizione (imputazione); le misure vere restano invariate.
    vals[nan_mask] = preds_all[nan_mask, j]
    # Rimette la colonna completa nella tabella.
    df_full[col] = vals

# Punteggio di selettività per la Classe I di ogni molecola.
df_full['score'] = selectivity_score(df_full)[2]
# Salva il dataset completo (misure + imputazioni + punteggio).
df_full.to_excel(os.path.join(SCREENING_OUT, 'dataset_scored.xlsx'), index=False)

In [ ]:
# File della libreria esterna di controllo (composti amari, BitterDB).
BITTER_CSV = 'bitter.csv'
# Colonna che contiene gli SMILES in quel file.
SMILES_COL = 'canonical_smiles'

# Legge la libreria.
bitter = pd.read_csv(BITTER_CSV)
# Numero di righe.
print(f'bitter.csv: {len(bitter)} rows | SMILES column = "{SMILES_COL}"')


# Tiene il frammento più grande di uno SMILES (toglie sali e controioni, es. "molecola.Cl").
def largest_fragment(smi):
    # Interpreta lo SMILES.
    mol = Chem.MolFromSmiles(str(smi))
    # Non valido: None.
    if mol is None:
        # La riga verrà scartata.
        return None
    # Separa i frammenti non collegati tra loro.
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    # SMILES del frammento con più atomi pesanti.
    return Chem.MolToSmiles(max(frags, key=lambda m: m.GetNumHeavyAtoms())) if frags else None


# SMILES ripuliti.
bitter['smiles_clean'] = bitter[SMILES_COL].map(largest_fragment)
# Quanti non sono leggibili.
n_bad = int(bitter['smiles_clean'].isna().sum())
# Toglie le righe non leggibili.
bitter = bitter.dropna(subset=['smiles_clean']).reset_index(drop=True)
# Riepilogo.
print(f'unparsed SMILES discarded: {n_bad} | valid molecules: {len(bitter)}')

# Predizioni delle 9 isoforme con la catena completa (passo 7).
pred_df = predict_to_dataframe(bitter['smiles_clean'].tolist(), bundle, device=device)
# Punteggio di selettività di ogni composto, calcolato dalle predizioni (che non vengono salvate).
bitter['mean_class_I'], bitter['max_others'], bitter['score'] = selectivity_score(pred_df)
# Tiene solo SMILES, media Classe I, massimo delle altre isoforme e punteggio; ordina dal più selettivo.
bitter = (bitter[['smiles_clean', 'mean_class_I', 'max_others', 'score']]
          .sort_values('score', ascending=False).reset_index(drop=True))

# Percorso del file di output.
out_path = os.path.join(SCREENING_OUT, 'bitter_scores.csv')
# Salva i punteggi.
bitter.round(4).to_csv(out_path, index=False)
# Conferma.
print(f'saved -> {out_path}  ({len(bitter)} molecules)')
# Statistiche del punteggio.
print(f"selectivity score: mean={bitter['score'].mean():.2f}  max={bitter['score'].max():.2f}  "
      f"min={bitter['score'].min():.2f}  | compounds with score>0: {(bitter['score'] > 0).sum()}")


In [ ]:
# Dataset completo ordinato dal composto più selettivo.
df_full = df_full.sort_values('score', ascending=False).reset_index(drop=True)


# Salva come immagine una tabella dei primi n composti per punteggio.
def save_top_table(df, smiles_col, fname, title, n=15):
    # Primi n composti.
    d = df.sort_values('score', ascending=False).head(n).copy()
    # Componenti del punteggio, arrotondate.
    d['mean_class_I'], d['max_others'], _ = (v.round(2) for v in selectivity_score(d))
    # Punteggio arrotondato.
    d['score'] = d['score'].round(2)
    # SMILES accorciati a 30 caratteri.
    d[smiles_col] = d[smiles_col].astype(str).str.slice(0, 30)
    # Colonne della tabella…
    cols = [smiles_col, 'mean_class_I', 'max_others', 'score']
    # …e loro intestazioni.
    labels = ['SMILES', 'mean Class I', 'max other', 'score']
    # Figura alta quanto serve per n righe.
    fig, ax = plt.subplots(figsize=(9, 0.45 * (n + 1)))
    # Nessun asse: solo la tabella.
    ax.axis('off')
    # Titolo.
    ax.set_title(title, fontsize=12, weight='bold', pad=12)
    # Disegna la tabella.
    tbl = ax.table(cellText=d[cols].values, colLabels=labels, loc='center', cellLoc='center')
    # Testo a dimensione fissa e righe un po' più alte.
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.4)
    # Intestazione blu con testo bianco in grassetto.
    for j in range(len(cols)):
        # Cella j della riga di intestazione.
        cell = tbl[0, j]; cell.set_facecolor('#4472C4'); cell.set_text_props(color='white', weight='bold')
    # Sistema gli spazi.
    plt.tight_layout()
    # Salva l'immagine.
    plt.savefig(os.path.join(SCREENING_OUT, fname), bbox_inches='tight', dpi=150)
    # La mostra.
    plt.show()


# Top 15 del dataset completo.
save_top_table(df_full, 'Smiles', 'dataset_top15.png', 'Top 15 - df_full')

# Punteggi delle due librerie (senza NaN).
s_full = df_full['score'].dropna().to_numpy()  # dataset completo (il focus)
s_bitter = bitter['score'].dropna().to_numpy()  # libreria esterna (riferimento)

# Colori e tinte del grafico.
C_FULL, C_BITTER = '#2a78d6', '#eb6834'  # blu = dataset, arancione = bitter
INK, INK_SOFT, GRID = '#0b0b0b', '#52514e', '#e4e3df'  # testo, testo secondario, griglia

# Distribuzione cumulativa empirica: valori ordinati e frazione cumulata.
def ecdf(x):
    # Valori ordinati.
    xs = np.sort(x)
    # Frazione di valori <= ciascuno.
    return xs, np.arange(1, len(xs) + 1) / len(xs)


# Estremi comuni (limitati ai percentili 0.5-99.5 per non schiacciare il grafico sulle code)…
lo, hi = np.percentile(np.concatenate([s_full, s_bitter]), [0.5, 99.5])
# …e 40 intervalli uguali, identici per le due librerie.
bins = np.linspace(lo, hi, 41)

# Due riquadri con lo stesso asse x, sfondo chiaro.
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.5, 4.8), sharex=True, facecolor='#fcfcfb',
                               gridspec_kw={'wspace': 0.22})
for ax in (axL, axR):
    ax.set_facecolor('#fcfcfb')  # sfondo del riquadro
    ax.spines[['top', 'right', 'left']].set_visible(False)  # solo l'asse x resta visibile
    ax.spines['bottom'].set_color(GRID)  # asse x tenue
    ax.grid(axis='y', color=GRID, lw=0.8)  # griglia solo orizzontale, leggera
    ax.set_axisbelow(True)  # griglia dietro ai dati
    ax.tick_params(colors=INK_SOFT, length=0, labelsize=11)  # niente trattini, testo secondario
    ax.axvspan(0, hi, color=C_FULL, alpha=0.05, lw=0)  # zona "selettivo per la Classe I" (score > 0)
    ax.axvline(0, color=INK_SOFT, lw=1, ls=':')  # linea dello zero

# Sinistra: istogramma pieno per il dataset (focus), solo contorno per la libreria esterna.
axL.hist(s_full, bins=bins, density=True, color=C_FULL, alpha=0.85,
         edgecolor='#fcfcfb', linewidth=0.8, label=f'Dataset (n={len(s_full):,})', zorder=3)
axL.hist(s_bitter, bins=bins, density=True, histtype='step', color=C_BITTER,
         linewidth=2, label=f'Bitter (n={len(s_bitter):,})', zorder=4)
# Mediane: linee tratteggiate nel colore della serie, valore nel testo.
for s, c, side in [(s_full, C_FULL, -1), (s_bitter, C_BITTER, 1)]:
    # Mediana della libreria.
    med = np.median(s)
    # Linea verticale.
    axL.axvline(med, color=c, ls='--', lw=1.6, zorder=5)
    # Valore, in alto e sempre in inchiostro (non nel colore della serie).
    axL.annotate(f'median {med:.2f}', (med, 1.0), xycoords=('data', 'axes fraction'), xytext=(7 * side, 6),
                 textcoords='offset points', va='bottom', ha='left' if side > 0 else 'right',
                 fontsize=10.5, color=INK_SOFT, annotation_clip=False)
axL.set_ylabel('Density', color=INK_SOFT, fontsize=12)  # asse y
axL.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 0.86), fontsize=11, labelcolor=INK)  # legenda

# Destra: distribuzioni cumulative, linee sottili a gradini.
for s, c in [(s_full, C_FULL), (s_bitter, C_BITTER)]:
    # Valori ordinati e frazione cumulata.
    xs, ys = ecdf(s)
    # Curva a gradini.
    axR.step(xs, ys, where='post', color=c, lw=2.2, zorder=3)
# Quota di composti con score > 0 del dataset (il focus), scritta nel grafico.
axR.annotate(f'{100 * (s_full > 0).mean():.0f}% of dataset\nhas score > 0', (0, np.mean(s_full <= 0)),
             xytext=(14, -6), textcoords='offset points', va='top', fontsize=10.5, color=INK_SOFT,
             arrowprops=dict(arrowstyle='-', color=INK_SOFT, lw=0.8))
axR.set_ylim(0, 1)  # una frazione va da 0 a 1
axR.set_ylabel('Cumulative fraction', color=INK_SOFT, fontsize=12)  # etichetta

# Limiti dell'asse x e etichetta comune.
axL.set_xlim(lo, hi)  # vale per entrambi i riquadri (sharex)
fig.supxlabel('Class I selectivity score  (mean Class I pIC50 − max other isoform)', color=INK_SOFT, fontsize=12)
fig.suptitle('Selectivity score: dataset vs bitter compounds', x=0.07, ha='left', fontsize=15,
             weight='bold', color=INK)
# Salva l'immagine.
plt.savefig(os.path.join(SCREENING_OUT, 'score_comparison.png'), bbox_inches='tight', dpi=200, facecolor=fig.get_facecolor())
# La mostra.
plt.show()


# Riga di statistiche per una libreria.
def summ(name, s):
    # Media, mediana, estremi e quota di composti selettivi (punteggio > 0).
    print(f'{name:8s}: mean={s.mean():.2f}  median={np.median(s):.2f}  max={s.max():.2f}  min={s.min():.2f}  '
          f'| score>0: {(s > 0).sum()} ({100 * (s > 0).mean():.1f}%)')


# Statistiche delle due librerie.
print('--- Class I selectivity score ---')  # intestazione
summ('df_full', s_full)  # dataset completo
summ('bitter', s_bitter)  # libreria esterna

In [ ]:
# Gruppi chimici noti per legare lo zinco del sito attivo delle HDAC (Zinc Binding Groups),
# scritti come pattern SMARTS. L'ordine conta: per ogni molecola vale il primo trovato.
ZBG_SMARTS = {
    'hydroxamic acid (pan-HDAC)': '[CX3](=[OX1])[NX3][OX2H1]',
    'aminoanilide (Class I)': '[CX3](=[OX1])[NX3]c1ccccc1[NX3;H1,H2]',
    'mercaptoacetamide': '[SX2H][CH2][CX3](=[OX1])[NX3]',
    '2-mercaptobenzamide': '[SX2H]c1ccccc1[CX3](=[OX1])[NX3]',
    'alpha-ketoamide': '[CX3](=[OX1])[CX3](=[OX1])[NX3]',
    'trifluoromethyl ketone': '[CX3](=[OX1])C(F)(F)F',
    'difluoromethyl ketone': '[CX3](=[OX1])[CX4](F)F',
    'acylhydrazide': '[CX3](=[OX1])[NX3][NX3]',
    'oxadiazol-5-one': 'c1noc(=O)[nH]1',
    'hydroxypyridinethione (HOPTO)': '[OX2H]c1ccc[nX2]c1[SX2H]',
    'tropolone': 'Oc1cccccc1=O',
    'squaramide': '[#8]=c1c([#7])c([#7])c1=[#8]',
    '8-hydroxyquinoline': '[OX2H]c1cccc2cccnc12',
    'catechol': 'c1cc(O)c(O)cc1',
    'boronic acid': '[BX3]([OX2H])[OX2H]',
    'thiol': '[SX2H]',
    'carboxylic acid (weak)': '[CX3](=[OX1])[OX2H1]',
}
# Pattern già interpretati da RDKit, pronti per la ricerca.
ZBG_PATT = {k: Chem.MolFromSmarts(v) for k, v in ZBG_SMARTS.items()}


# Primo gruppo noto presente nella molecola: (nome, atomi coinvolti), oppure (None, []).
def zbg_hit(mol):
    # Prova i gruppi nell'ordine del dizionario.
    for name, patt in ZBG_PATT.items():
        # Atomi della molecola che corrispondono al pattern (vuoto se assente).
        match = mol.GetSubstructMatch(patt)
        # Al primo trovato…
        if match:
            # …restituisce nome e atomi.
            return name, list(match)
    # Nessun gruppo trovato.
    return None, []


# Descrittori di "drug-likeness" e numero di violazioni della regola di Lipinski.
def judge(mol):
    # Peso molecolare e lipofilia.
    mw, logp = Descriptors.MolWt(mol), Crippen.MolLogP(mol)
    # Donatori e accettori di legame a idrogeno.
    hbd, hba = Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol)
    # Violazioni di Lipinski (0 = profilo da farmaco orale).
    viol = int((mw > 500) + (logp > 5) + (hbd > 5) + (hba > 10))
    # Restituisce i descrittori arrotondati.
    return {'MW': round(mw, 1), 'logP': round(logp, 2), 'HBD': hbd, 'HBA': hba,
            'TPSA': round(Descriptors.TPSA(mol), 1), 'RotB': Descriptors.NumRotatableBonds(mol),
            'Lipinski_viol': viol}


# Nome del gruppo noto di uno SMILES (None se assente o SMILES non valido).
def zbg_of_smiles(smi):
    # Interpreta lo SMILES.
    m = Chem.MolFromSmiles(str(smi))
    # Nome del primo gruppo trovato.
    return None if m is None else zbg_hit(m)[0]


# Dataset completo ordinato per punteggio, con il gruppo noto di ogni molecola.
work = df_full.sort_values('score', ascending=False).copy()
# Gruppo noto di ogni molecola (None se assente).
work['ZBG'] = work['Smiles'].map(zbg_of_smiles)

# Quante molecole hanno un gruppo noto, in totale e tra le prime 20.
n_lib, n_hit = len(work), int(work['ZBG'].notna().sum())
# Quota sul totale.
print(f'df_full: {n_hit}/{n_lib} molecules ({100 * n_hit / n_lib:.1f}%) carry a known group')
# Quota tra le prime 20 per punteggio.
print(f'among the top-20 by score: {int(work.head(20)["ZBG"].notna().sum())}/20 carry a known group')

# Le prime 20 molecole per punteggio che hanno un gruppo noto.
hit = work[work['ZBG'].notna()].head(20).reset_index(drop=True)
# Componenti del punteggio.
hit['mean_class_I'], hit['max_others'], _ = selectivity_score(hit)
# Riepilogo dei gruppi trovati.
print(f'\nshowing {len(hit)} molecules (top by score carrying a known group)')
# Intestazione…
print('group distribution among the selected molecules:')
# …e conteggio di ogni gruppo.
print(hit['ZBG'].value_counts().to_string())

# Molecole, didascalie, atomi da evidenziare e righe della tabella finale.
mols, legends, highlights, rows = [], [], [], []
# Per ogni molecola selezionata…
for i, r in hit.iterrows():
    # …la molecola RDKit…
    m = Chem.MolFromSmiles(str(r['Smiles']))
    # …il suo gruppo e i suoi atomi…
    name, atoms = zbg_hit(m)
    # …accumulati per il disegno…
    mols.append(m)  # struttura
    highlights.append(atoms)  # atomi del gruppo da colorare
    legends.append(f"#{i + 1}  s={r['score']:.2f}  {name}")  # didascalia: posizione, punteggio, gruppo
    # …e la riga della tabella con punteggio e drug-likeness.
    rows.append({'rank': i + 1, 'score': round(r['score'], 2), 'mean_class_I': round(r['mean_class_I'], 2),
                 'max_others': round(r['max_others'], 2), 'ZBG': name, **judge(m), 'SMILES': str(r['Smiles'])[:55]})

# Griglia di strutture con il gruppo evidenziato.
img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(260, 220), legends=legends,
                           highlightAtomLists=highlights, useSVG=False)
# La mostra…
display(img)

# Tabella finale delle 20 molecole.
display(pd.DataFrame(rows))